# Pré-processamento do conjunto de dados de galáxias

Este notebook foi organizado em funções documentadas, mantendo a lógica original: carregar os dados brutos do IRSA, remover linhas com valores ausentes em colunas-chave, padronizar (escalar) as features e salvar o conjunto de dados final em `galaxies.csv`.


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler


## 1. Carregamento dos dados brutos

In [2]:
def load_raw_dataset(path: str) -> pd.DataFrame:
    """
    Carrega o conjunto de dados bruto (resultado da busca no catálogo IRSA)
    a partir de um arquivo CSV.

    Parameters
    ----------
    path : str
        Caminho do arquivo CSV contendo os dados brutos.

    Returns
    -------
    pandas.DataFrame or None
        DataFrame contendo todas as amostras, ou ``None`` caso o
        arquivo não seja encontrado.
    """

    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        print("Arquivo não encontrado...")
        return None


In [3]:
df_raw = load_raw_dataset("table_irsa_catalog_search_results.csv")
df_raw.head()


,sequentialid,ra,dec,gg,m20,cc,aa,r_gim2d,ell_gim2d,sersic_n_gim2d,type
0,87784.0,149.904022,2.386357,0.499754,-2.084311,3.237597,0.013838,NaN,NaN,NaN,2.0
1,87785.0,149.906021,2.400651,0.474288,-1.557100,2.896190,0.068762,0.596900,0.72999,1.0144,2.0
2,87787.0,149.909744,2.422317,0.430558,-1.608219,2.577024,-0.040027,NaN,NaN,NaN,2.0
3,87788.0,149.902328,2.379942,0.443384,-1.857250,2.765364,0.058141,NaN,NaN,NaN,2.0
4,87789.0,149.910294,2.426711,0.516209,-1.969563,2.983260,0.057431,0.244165,0.74957,2.4039,2.0


## 2. Verificação de valores ausentes

In [4]:
def report_missing_values(df: pd.DataFrame, columns: list) -> pd.Series:
    """
    Exibe e retorna a contagem de valores ausentes (``NaN``) para um
    conjunto de colunas específico.

    Parameters
    ----------
    df : pandas.DataFrame
        Conjunto de dados a ser inspecionado.
    columns : list of str
        Lista com os nomes das colunas a serem verificadas.

    Returns
    -------
    pandas.Series
        Série indexada pelo nome da coluna, com a contagem de valores
        ausentes em cada uma.
    """

    contagem_nan = df[columns].isna().sum()
    print(contagem_nan)
    return contagem_nan


In [5]:
colunas_criticas = ["r_gim2d", "ell_gim2d", "sersic_n_gim2d"]

_ = report_missing_values(df_raw, colunas_criticas)
print(df_raw.shape)


r_gim2d           73158
ell_gim2d         73158
sersic_n_gim2d    73158
dtype: int64
(108032, 11)


## 3. Remoção de linhas com valores ausentes

In [6]:
def drop_missing_rows(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Remove as linhas que possuem valores ausentes (NaN) em
    qualquer uma das colunas informadas, reindexando o resultado e
    exibindo um resumo de quantas linhas foram removidas.

    Parameters
    ----------
    df : pandas.DataFrame
        Conjunto de dados original.
    columns : list of str
        Lista com os nomes das colunas consideradas para a remoção de
        linhas (uma linha é removida se tiver ``NaN`` em qualquer uma
        delas).

    Returns
    -------
    pandas.DataFrame
        Novo DataFrame, sem as linhas com valores ausentes nas colunas
        informadas, com o índice reiniciado.
    """

    linhas_antes = len(df)

    dataset = df.dropna(subset=columns).reset_index(drop=True)

    linhas_depois = len(dataset)

    print(f"Linhas antes: {linhas_antes}")
    print(f"Linhas depois: {linhas_depois}")
    print(f"Linhas removidas: {linhas_antes - linhas_depois}")

    return dataset


In [7]:
dataset = drop_missing_rows(df_raw, colunas_criticas)

dataset.shape


Linhas antes: 108032
Linhas depois: 34874
Linhas removidas: 73158


(34874, 11)

In [8]:
dataset.head()


,sequentialid,ra,dec,gg,m20,cc,aa,r_gim2d,ell_gim2d,sersic_n_gim2d,type
0,87785.0,149.906021,2.400651,0.474288,-1.557100,2.896190,0.068762,0.596900,0.72999,1.0144,2.0
1,87789.0,149.910294,2.426711,0.516209,-1.969563,2.983260,0.057431,0.244165,0.74957,2.4039,2.0
2,87792.0,149.906845,2.414332,0.541874,-2.142101,3.391024,0.089022,0.392615,0.55932,3.7844,2.0
3,87794.0,149.902878,2.394248,0.587865,-2.156702,3.422536,0.065432,0.305705,0.44282,4.4370,1.0
4,87795.0,149.903214,2.398019,0.503267,-2.001612,3.203920,0.103682,0.535650,0.59589,2.7911,2.0


## 4. Seleção de atributos (features) e rótulo (target)

In [9]:
def select_features_and_target(
    dataset: pd.DataFrame,
    feature_cols: list,
    target_cols: list,
):
    """
    Separa o conjunto de dados em atributos (features) e rótulo
    (target), a partir de listas explícitas de colunas.

    Parameters
    ----------
    dataset : pandas.DataFrame
        Conjunto de dados já limpo (sem valores ausentes nas colunas
        relevantes).
    feature_cols : list of str
        Nomes das colunas utilizadas como atributos (features) do
        modelo.
    target_cols : list of str
        Nomes das colunas mantidas como identificação/rótulo (por
        exemplo, "sequentialid" e "type").

    Returns
    -------
    tuple of (pandas.DataFrame, pandas.DataFrame)
        Uma tupla ``(X, y)`` em que ``X`` contém as features e ``y``
        contém as colunas de identificação/rótulo.
    """

    X = dataset[feature_cols]
    y = dataset[target_cols]

    return X, y


In [10]:
colunas_features = [
    "gg",
    "m20",
    "cc",
    "aa",
    "sersic_n_gim2d",
    "ell_gim2d",
    "r_gim2d",
]
colunas_target = ["sequentialid", "type"]

X, y = select_features_and_target(
    dataset,
    feature_cols=colunas_features,
    target_cols=colunas_target,
)


## 5. Padronização (scaling) das features

In [11]:
def scale_features(X: pd.DataFrame, scaler: StandardScaler = None):
    """
    Padroniza (normaliza) as features utilizando StandardScaler
    (média 0, desvio-padrão 1).

    Se um scaler já ajustado for informado, ele é reutilizado
    apenas para transformar os dados (transform), o que é útil ao
    aplicar a mesma padronização a novos dados (por exemplo, dados de
    teste ou de produção). Caso contrário, um novo StandardScaler
    é ajustado aos dados (fit_transform).

    Parameters
    ----------
    X : pandas.DataFrame
        Atributos (features) a serem padronizados.
    scaler : sklearn.preprocessing.StandardScaler, optional
        Scaler já ajustado a ser reutilizado. Se None (padrão), um
        novo scaler é criado e ajustado a X.

    Returns
    -------
    tuple of (pandas.DataFrame, sklearn.preprocessing.StandardScaler)
        Uma tupla (X_scaled, scaler) em que X_scaled é o
        DataFrame com as features padronizadas (mesmas colunas e
        índice de X) e scaler é a instância do
        StandardScaler utilizada (ajustada ou reutilizada).
    """

    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    X_scaled = pd.DataFrame(
        X_scaled,
        columns=X.columns,
        index=X.index,
    )

    return X_scaled, scaler


In [12]:
X_scaled, scaler = scale_features(X)

print(X_scaled.head())
print(X_scaled.describe())


         gg       m20        cc        aa  sersic_n_gim2d  ell_gim2d   r_gim2d
0 -0.178659  0.408056  0.195886 -0.090374       -0.476385   1.510386  0.014649
1  0.542650 -0.742287  0.370185 -0.287801        0.546606   1.602553 -0.936228
2  0.984253 -1.223489  1.186458  0.262630        1.562971   0.707001 -0.536047
3  1.775592 -1.264211  1.249540 -0.148394        2.043435   0.158608 -0.770333
4  0.319965 -0.831671  0.811909  0.518061        0.831674   0.879145 -0.150464
                 gg           m20            cc            aa  sersic_n_gim2d  \
count  3.487400e+04  3.487400e+04  3.487400e+04  3.487400e+04    3.487400e+04   
mean  -5.576520e-16 -2.473473e-16  9.739044e-17  8.149828e-17    4.920459e-17   
std    1.000014e+00  1.000014e+00  1.000014e+00  1.000014e+00    1.000014e+00   
min   -3.034062e+00 -2.886544e+00 -2.763399e+00 -4.614986e+00   -1.075971e+00   
25%   -7.752913e-01 -7.447192e-01 -7.374220e-01 -6.160674e-01   -6.825001e-01   
50%   -8.869558e-02 -2.778893e-02 -1.318

## 6. Montagem e exportação do conjunto de dados final

In [13]:
def build_final_dataset(
    target_df: pd.DataFrame,
    scaled_features_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Combina as colunas de identificação/rótulo com as features já
    padronizadas, formando o conjunto de dados final.

    Parameters
    ----------
    target_df : pandas.DataFrame
        Colunas de identificação/rótulo (ex.: sequentialid,type).
    scaled_features_df : pandas.DataFrame
        Features já padronizadas (mesmo índice de target_df).

    Returns
    -------
    pandas.DataFrame
        DataFrame final, com as colunas de identificação/rótulo
        seguidas pelas features padronizadas.
    """

    return pd.concat([target_df, scaled_features_df], axis=1)


In [14]:
galaxies = build_final_dataset(y, X_scaled)

galaxies.head()


,sequentialid,type,gg,m20,cc,aa,sersic_n_gim2d,ell_gim2d,r_gim2d
0,87785.0,2.0,-0.178659,0.408056,0.195886,-0.090374,-0.476385,1.510386,0.014649
1,87789.0,2.0,0.542650,-0.742287,0.370185,-0.287801,0.546606,1.602553,-0.936228
2,87792.0,2.0,0.984253,-1.223489,1.186458,0.262630,1.562971,0.707001,-0.536047
3,87794.0,1.0,1.775592,-1.264211,1.249540,-0.148394,2.043435,0.158608,-0.770333
4,87795.0,2.0,0.319965,-0.831671,0.811909,0.518061,0.831674,0.879145,-0.150464


In [15]:
def save_dataset(df: pd.DataFrame, path: str) -> None:
    """
    Salva o DataFrame final em um arquivo CSV.

    Parameters
    ----------
    df : pandas.DataFrame
        Conjunto de dados a ser salvo.
    path : str
        Caminho (nome do arquivo) onde o CSV será salvo.

    Returns
    -------
    None
    """

    df.to_csv(path, index=False)
    print(f"Conjunto de dados salvo em: {path}")


In [16]:
save_dataset(galaxies, "galaxies.csv")


Conjunto de dados salvo em: galaxies.csv
